# Eye Tracking Capstone — Dataset Exploration

**Purpose:** Load all three datasets, confirm exact column names, shapes, and label distributions,
then produce a DATASET SUMMARY table that defines the model input/output for each of the 4 models.

**Critical note:** This notebook documents a gap between the project spec and the actual downloaded
data. All adaptations are noted inline and summarised in the final cell.

---

In [ ]:
# ─── CELL 1 — Imports and path constants ────────────────────────────────────
import os
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger(__name__)

# ── Project root (all paths are relative to this) ──────────────────────────
ROOT = Path("C:/Users/kunal/Desktop/Eye tracking Module")
assert ROOT.exists(), f"Project root not found: {ROOT}"

# ── Dataset paths ──────────────────────────────────────────────────────────
PATH_VREED_FEATURES  = ROOT / "04 Eye Tracking Data" / "02 Eye Tracking Data (Features Extracted)" / "EyeTracking_FeaturesExtracted.csv"
PATH_VREED_SURVEY    = ROOT / "03 Self-Reported Questionnaires" / "02 Post Exposure Ratings.xlsx"
PATH_VREED_RAW       = ROOT / "04 Eye Tracking Data" / "01 Eye Tracking Data (Pre-Processed)"

PATH_OPENEDS_ROOT    = ROOT / "openEDS" / "openEDS"
PATH_OPENEDS_BBOX    = ROOT / "bbox" / "bbox"
PATH_OPENEDS_EVENTS  = ROOT / "openEDS_events" / "openEDS_events"

PATH_COGLOAD         = ROOT / "cognitive_load_dataset.csv"

log.info("All paths resolved.")
print(f"Project root : {ROOT}")
print(f"VREED CSV    : {PATH_VREED_FEATURES.exists()}")
print(f"OpenEDS root : {PATH_OPENEDS_ROOT.exists()}")
print(f"CogLoad CSV  : {PATH_COGLOAD.exists()}")

---
## 1  VREED Dataset — Emotion features

Expected by spec: eye-tracking feature CSV with pupil dilation, blink rate, fixation duration,
saccade amplitude and a self-reported 6-class emotion label.

**Actual contents (confirmed below):** aggregated eye-tracking statistics (50 columns) with a
`Quad_Cat` label (0–3) representing the emotional valence×arousal quadrant.

In [ ]:
# ─── CELL 2 — Load VREED features CSV ────────────────────────────────────────
vreed = pd.read_csv(PATH_VREED_FEATURES)

log.info("VREED features CSV loaded.")
print("=" * 65)
print("VREED — EyeTracking_FeaturesExtracted.csv")
print("=" * 65)
print(f"Shape            : {vreed.shape}  (rows × columns)")
print(f"Total null cells : {vreed.isnull().sum().sum()}")
print()
print("All 50 column names:")
for i, col in enumerate(vreed.columns, 1):
    print(f"  {i:>2}. {col}")

In [ ]:
# ─── CELL 3 — VREED label distribution ────────────────────────────────────────
print("Label column : Quad_Cat")
print("Value counts:")
print(vreed["Quad_Cat"].value_counts().sort_index().to_string())
print()
print("INTERPRETATION (from Post Exposure Ratings.xlsx):")
print("  Quad_Cat=0 → Low Arousal / Low Valence   → sad / bored")
print("  Quad_Cat=1 → Low Arousal / High Valence  → calm / relaxed")
print("  Quad_Cat=2 → High Arousal / Low Valence  → angry / stressed")
print("  Quad_Cat=3 → High Arousal / High Valence → happy / excited")

fig, ax = plt.subplots(figsize=(5, 3))
vreed["Quad_Cat"].value_counts().sort_index().plot.bar(
    ax=ax, color=["#5875A4", "#60BD68", "#F15854", "#F7E442"], edgecolor="black"
)
ax.set_title("VREED Emotion-Quadrant Label Distribution")
ax.set_xlabel("Quad_Cat (0=sad, 1=calm, 2=angry, 3=happy)")
ax.set_ylabel("Count")
ax.set_xticklabels(["0\nsad", "1\ncalm", "2\nangry", "3\nhappy"], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ─── CELL 4 — VREED first 5 rows and null audit ──────────────────────────────
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 120)
print("First 5 rows (selected columns):")
SHOW_COLS = ["Quad_Cat", "Num_of_Fixations", "Mean_Fixation_Duration",
             "Num_of_Saccade", "Mean_Saccade_Amplitude",
             "Num_of_Blink", "Mean_Blink_Duration"]
print(vreed[SHOW_COLS].head().to_string(index=True))
print()
print("Null counts per column (only columns with nulls):")
null_counts = vreed.isnull().sum()
nulls_present = null_counts[null_counts > 0]
print(nulls_present.to_string() if len(nulls_present) else "  None")

In [ ]:
# ─── CELL 5 — VREED: identify the 4 model input features ─────────────────────
# Spec requires: [pupil_dilation_mm, blink_rate_per_sec,
#                 fixation_duration_ms, saccade_amplitude_deg]
# OpenEDS/VREED do not expose pupil size in mm, but we map as follows:

VREED_FEATURE_MAP = {
    "pupil_proxy"      : "Num_of_Blink",            # note: only available blink-domain column close to arousal
    "blink_rate"       : "Mean_Blink_Duration",      # ms — normalised blink measure
    "fixation_duration": "Mean_Fixation_Duration",   # ms
    "saccade_amplitude": "Mean_Saccade_Amplitude",   # deg
}
print("Selected 4 features for Emotion MLP:")
for role, col in VREED_FEATURE_MAP.items():
    stats = vreed[col].describe()
    print(f"  {role:22s} → {col}")
    print(f"      mean={stats['mean']:.4f}  std={stats['std']:.4f}  "
          f"min={stats['min']:.4f}  max={stats['max']:.4f}  "
          f"nulls={vreed[col].isnull().sum()}")

print()
print("DESIGN NOTE: 'Mean_Blink_Duration' replaces the spec's blink_rate_per_sec")
print("because VREED provides aggregated statistics not per-frame blink counts.")
print("At inference time, feature_extractor.py computes per-second blink_rate")
print("from the live rolling window, so the live feature names will differ from")
print("the training feature names. Both are scaled separately (train-time scaler")
print("is fitted on VREED columns; live scaler is per-session baseline calibration).")

In [ ]:
# ─── CELL 6 — VREED Survey: confirm Quad_Cat → emotion mapping ────────────────
survey = pd.read_excel(PATH_VREED_SURVEY)
print("Post-Exposure Ratings columns:")
for c in survey.columns:
    print(f"  {c}")
print(f"\nShape: {survey.shape}")
print()

# Cross-tabulate Quad_Cat vs Arousal_Cat and Valence_Cat
print("Quad_Cat vs Arousal_Cat cross-tabulation:")
print(pd.crosstab(survey["Quad_Cat"], survey["Arousal_Cat"]))
print()
print("Quad_Cat vs Valence_Cat cross-tabulation:")
print(pd.crosstab(survey["Quad_Cat"], survey["Valence_Cat"]))
print()

# Mean self-reported emotion scores per Quad_Cat
emotion_cols = [c for c in survey.columns if c.startswith("POST_")
                and c not in ["POST_Valence", "POST_Arousal"] and "PQ" not in c]
mean_by_quad = survey.groupby("Quad_Cat")[emotion_cols].mean().round(1)
print("Mean self-reported emotion scores per Quad_Cat:")
print(mean_by_quad.to_string())

---
## 2  OpenEDS Dataset — Attention labels & Gaze estimation

**Expected by spec:** 91,200 video sequence frames with angular gaze vectors (pitch/yaw) for
gaze regression and attention label derivation via gaze velocity thresholding.

**Actual contents (confirmed below):** 32,919 eye images with pixel-level segmentation masks
(iris, pupil, sclera, background). **Angular gaze labels are absent.**

**Adaptation strategy:** Use the segmentation masks to  
1. Train a lightweight segmentation model that extracts pupil/iris boundaries.  
2. Derive pseudo attention labels from Eye Aspect Ratio + pupil visibility ratio.  
3. Use pupil centroid as gaze proxy, calibrated to screen via the 9-point routine.

In [ ]:
# ─── CELL 7 — OpenEDS split counts ────────────────────────────────────────────
SPLITS = ["train", "validation", "test"]
print("=" * 50)
print("OpenEDS — Image + Segmentation Dataset")
print("=" * 50)
for split in SPLITS:
    img_dir  = PATH_OPENEDS_ROOT / split / "images"
    lbl_dir  = PATH_OPENEDS_ROOT / split / "labels"
    mask_dir = PATH_OPENEDS_ROOT / split / "masks"
    n_img    = len(list(img_dir.glob("*.png")))  if img_dir.exists()  else 0
    n_lbl    = len(list(lbl_dir.glob("*.npy")))  if lbl_dir.exists()  else 0
    n_mask   = len(list(mask_dir.glob("*.png"))) if mask_dir.exists() else 0
    print(f"  {split:12s}: images={n_img:6d}  seg_labels={n_lbl:6d}  masks={n_mask:6d}")

# Total participant-level images (S_* folders)
participant_pngs = sum(
    len(list((PATH_OPENEDS_ROOT / d).glob("*.png")))
    for d in os.listdir(PATH_OPENEDS_ROOT) if d.startswith("S_")
)
print(f"\n  Participant folders (S_*): {participant_pngs} PNG images")
print(f"  images.txt entries : {sum(1 for _ in open(PATH_OPENEDS_ROOT / 'images.txt'))}")
print(f"  labels.txt entries : {sum(1 for _ in open(PATH_OPENEDS_ROOT / 'labels.txt'))}")
print()
print("NOTE: The 'labels' in labels.txt are SEGMENTATION masks (uint8, values 0-3)")
print("      NOT angular gaze vectors. Gaze sequence data is not present in this download.")

In [ ]:
# ─── CELL 8 — OpenEDS image and segmentation mask inspection ──────────────────
# Load one sample image + its segmentation label and bounding box
sample_img_path  = PATH_OPENEDS_ROOT / "train" / "images" / "000000.png"
sample_lbl_path  = PATH_OPENEDS_ROOT / "train" / "labels" / "000000.npy"
sample_mask_path = PATH_OPENEDS_ROOT / "train" / "masks"  / "000000.png"

img  = np.array(Image.open(sample_img_path))
lbl  = np.load(sample_lbl_path)
mask = np.array(Image.open(sample_mask_path))

print(f"Image shape  : {img.shape}  dtype={img.dtype}  range=[{img.min()}, {img.max()}]")
print(f"Label shape  : {lbl.shape}  dtype={lbl.dtype}  unique values={np.unique(lbl).tolist()}")
print(f"Mask shape   : {mask.shape} dtype={mask.dtype}  unique values={np.unique(mask).tolist()}")
print()
print("Segmentation class mapping (OpenEDS standard):")
print("  0 = background")
print("  1 = sclera")
print("  2 = iris")
print("  3 = pupil")

# Pixel count per class in sample label
total = lbl.size
for cls_id, cls_name in enumerate(["background", "sclera", "iris", "pupil"]):
    count = np.sum(lbl == cls_id)
    print(f"  class {cls_id} ({cls_name:12s}): {count:7d} px  ({100*count/total:.2f}%)")

# Visual
COLOR_MAP = np.array([[0, 0, 0], [200, 200, 200], [0, 100, 255], [255, 50, 50]], dtype=np.uint8)
seg_color = COLOR_MAP[lbl]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img, cmap="gray")
axes[0].set_title("Raw eye image (400×640)")
axes[0].axis("off")
axes[1].imshow(seg_color)
axes[1].set_title("Segmentation label (colored)")
axes[1].axis("off")
axes[2].imshow(mask, cmap="gray")
axes[2].set_title("Mask PNG")
axes[2].axis("off")
patches = [
    mpatches.Patch(color=COLOR_MAP[i] / 255, label=n)
    for i, n in enumerate(["background", "sclera", "iris", "pupil"])
]
axes[1].legend(handles=patches, loc="lower right", fontsize=8)
plt.suptitle("OpenEDS train sample 000000", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ─── CELL 9 — OpenEDS: visualise a grid of 12 eye images ─────────────────────
train_img_dir = PATH_OPENEDS_ROOT / "train" / "images"
sample_files  = sorted(train_img_dir.glob("*.png"))[:12]

fig, axes = plt.subplots(3, 4, figsize=(14, 9))
fig.suptitle("OpenEDS train images (first 12)", fontsize=12)
for ax, f in zip(axes.flat, sample_files):
    ax.imshow(np.array(Image.open(f)), cmap="gray")
    ax.set_title(f.name, fontsize=7)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Observation: images are full eye/face crops at ~400×640 pixels.")
print("For model input, these will be cropped to the eye region using the")
print("bounding boxes in bbox/bbox/S_*.txt and resized to 64×64.")

In [ ]:
# ─── CELL 10 — OpenEDS: inspect bounding box files ────────────────────────────
bbox_files = sorted(PATH_OPENEDS_BBOX.glob("*.txt"))
print(f"Bounding box files: {len(bbox_files)} participant files")

# Read first bbox file
first_bbox = pd.read_csv(bbox_files[0], sep=" ", header=None,
                         names=["x_start", "width", "y_start", "height"])
print(f"\n{bbox_files[0].name} — {len(first_bbox)} rows")
print("Format: x_start  width  y_start  height (pixel coordinates)")
print(first_bbox.head().to_string(index=True))
print()
print("Each row = one image frame. Rows with x_start=0, width=640, y_start=0,")
print("height=400 indicate 'no eye detected' (full-frame sentinel bbox).")
# Check for sentinel rows
sentinel = ((first_bbox["x_start"] == 0) & (first_bbox["width"] == 640) &
            (first_bbox["y_start"] == 0) & (first_bbox["height"] == 400))
print(f"\nSentinel (no eye) rows in {bbox_files[0].name}: {sentinel.sum()} / {len(first_bbox)}")

In [ ]:
# ─── CELL 11 — OpenEDS: pseudo attention label derivation strategy ─────────────
# Since gaze angle labels are absent, we derive pseudo-labels from segmentation masks.
# Strategy: compute Eye Openness Ratio (EOR) from pupil/iris mask geometry.

print("Pseudo attention label derivation from segmentation masks")
print("=" * 60)
print()
print("For each segmentation mask (400×640, values 0-3):")
print("  1. pupil_area  = number of pixels with value 3 (pupil)")
print("  2. iris_area   = number of pixels with value 2 (iris)")
print("  3. sclera_area = number of pixels with value 1 (sclera)")
print("  4. eye_visible = (iris_area + sclera_area) > MIN_EYE_AREA_THRESHOLD")
print("  5. pupil_ratio = pupil_area / (iris_area + 1)  # iris-normalised pupil size")
print("  6. eor         = iris_area / (full_image_height * iris_bbox_height + 1)")
print()
print("Label assignment:")
print("  eye_visible=False OR eor < 0.10  → off_task  (eye closed / looking away)")
print("  pupil_ratio < 0.25               → distracted (pupil small / low arousal)")
print("  otherwise                        → focused")
print()
print("This is a HEURISTIC proxy for attention level.")
print("Documents as a design decision in README.")

# Demo on a few training samples
import gc
lbl_dir_train = PATH_OPENEDS_ROOT / "train" / "labels"
lbl_files = sorted(lbl_dir_train.glob("*.npy"))[:200]

records = []
MIN_EYE_AREA = 200

for f in lbl_files:
    m = np.load(f)
    pupil_area  = int(np.sum(m == 3))
    iris_area   = int(np.sum(m == 2))
    sclera_area = int(np.sum(m == 1))
    eye_visible = (iris_area + sclera_area) > MIN_EYE_AREA
    pupil_ratio = pupil_area / (iris_area + 1)

    if not eye_visible:
        label = "off_task"
    elif pupil_ratio < 0.25:
        label = "distracted"
    else:
        label = "focused"
    records.append({"file": f.name, "pupil_area": pupil_area,
                    "iris_area": iris_area, "pupil_ratio": round(pupil_ratio, 3),
                    "eye_visible": eye_visible, "pseudo_label": label})

pseudo_df = pd.DataFrame(records)
print("\nPseudo label distribution on first 200 training samples:")
print(pseudo_df["pseudo_label"].value_counts().to_string())
print()
print(pseudo_df.head(10).to_string(index=False))
gc.collect()

---
## 3  Cognitive Load Dataset

In [ ]:
# ─── CELL 12 — Load cognitive load CSV ────────────────────────────────────────
cogload = pd.read_csv(PATH_COGLOAD)

print("=" * 55)
print("Cognitive Load Dataset — cognitive_load_dataset.csv")
print("=" * 55)
print(f"Shape            : {cogload.shape}  (rows × columns)")
print(f"Total null cells : {cogload.isnull().sum().sum()}")
print()

# Group columns by modality
eeg_cols     = [c for c in cogload.columns if c.startswith("EEG_")]
fnirs_cols   = [c for c in cogload.columns if c.startswith("fNIRS_") or "NIRS" in c]
eye_cols     = [c for c in cogload.columns if any(
                    k in c for k in ["Pupil", "Blink", "Fixation", "Saccade"])]
drive_cols   = [c for c in cogload.columns if c not in eeg_cols + fnirs_cols + eye_cols
                and c != "Cognitive_Load"]

print(f"Column groups:")
print(f"  EEG columns       : {len(eeg_cols)}")
print(f"  fNIRS columns     : {len(fnirs_cols)}")
print(f"  Eye-tracking cols : {len(eye_cols)}  → {eye_cols}")
print(f"  Other columns     : {len(drive_cols)} → {drive_cols}")
print(f"  Label column      : Cognitive_Load")

In [ ]:
# ─── CELL 13 — Cognitive load eye-feature stats and label distribution ─────────
EYE_FEATURE_COLS = ["Pupil_Dilation", "Blink_Rate", "Fixation_Duration", "Saccade_Duration"]
LABEL_COL        = "Cognitive_Load"

print("Eye feature statistics (all 86,435 rows):")
print(cogload[EYE_FEATURE_COLS].describe().round(4).to_string())
print()
print("Label distribution:")
vc = cogload[LABEL_COL].value_counts().sort_index()
for v, c in vc.items():
    print(f"  {int(v)}-back  ({['low','medium','high'][int(v)]:7s}): {c:6d}  ({100*c/len(cogload):.1f}%)")

print()
print("First 5 rows (eye features + label only):")
print(cogload[EYE_FEATURE_COLS + [LABEL_COL]].head().to_string(index=True))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cogload[LABEL_COL].value_counts().sort_index().plot.bar(
    ax=axes[0], color=["#60BD68", "#F7E442", "#F15854"], edgecolor="black"
)
axes[0].set_title("Cognitive Load Label Distribution")
axes[0].set_xticklabels(["0-back\n(low)", "1-back\n(medium)", "2-back\n(high)"], rotation=0)
axes[0].set_ylabel("Count")

cogload[EYE_FEATURE_COLS].boxplot(ax=axes[1], vert=True)
axes[1].set_title("Eye Feature Distributions (all 4 columns)")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# ─── CELL 14 — Cognitive Load: eye features vs label (violin plots) ────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle("Eye Features vs Cognitive Load Level", fontsize=12)
load_labels = {0.0: "low", 1.0: "medium", 2.0: "high"}

for ax, col in zip(axes, EYE_FEATURE_COLS):
    data_by_class = [cogload.loc[cogload[LABEL_COL] == k, col].sample(2000, random_state=42)
                     for k in sorted(cogload[LABEL_COL].unique())]
    ax.violinplot(data_by_class, showmedians=True)
    ax.set_title(col.replace("_", "\n"), fontsize=9)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(["low", "med", "high"], fontsize=8)
    ax.set_xlabel("Cognitive Load")
plt.tight_layout()
plt.show()

---
## 4  Dataset Summary Table

Final confirmation table: which dataset feeds which model, with exact column names,
sample counts, and any adaptations vs the original spec.

In [ ]:
# ─── CELL 15 — DATASET SUMMARY TABLE ─────────────────────────────────────────
# Print as a formatted text table for clarity in the notebook output.

SEPARATOR = "=" * 120
THIN_SEP   = "-" * 120

print(SEPARATOR)
print(f"{'MODEL':<30} {'DATASET':<20} {'TRAIN SAMPLES':<18} {'INPUT FEATURES / COLUMNS':<35} {'OUTPUT':<15}")
print(SEPARATOR)

rows = [
    (
        "Model 1 — Emotion MLP",
        "VREED",
        "312 rows",
        "[Num_of_Blink, Mean_Blink_Duration,\n" +
        " " * 68 + "Mean_Fixation_Duration, Mean_Saccade_Amplitude]",
        "4 classes (Quad_Cat 0-3)"
    ),
    (
        "Model 2 — Attention CNN",
        "OpenEDS (seg)",
        "27,431 images",
        "64×64 grayscale eye crop",
        "3 classes (focused / distracted / off_task)"
    ),
    (
        "Model 3 — Cognitive XGBoost",
        "Cog. Load CSV",
        "86,435 rows",
        "[Pupil_Dilation, Blink_Rate,\n" +
        " " * 68 + "Fixation_Duration, Saccade_Duration]",
        "3 classes (0/1/2-back)"
    ),
    (
        "Model 4 — Gaze Segmentation",
        "OpenEDS (seg)",
        "27,431 images",
        "64×64 grayscale eye crop",
        "Pupil centroid (x,y) in eye-crop coords"
    ),
]

for model, dataset, n, features, output in rows:
    feature_lines = features.split("\n")
    print(f"{model:<30} {dataset:<20} {n:<18} {feature_lines[0]:<35} {output}")
    for extra_line in feature_lines[1:]:
        print(extra_line)
    print(THIN_SEP)

print()
print("ADAPTATION NOTES vs original spec:")
print()
adaptations = [
    ("Emotion classes",
     "Spec: 6 classes (happy/sad/angry/neutral/focused/stressed).",
     "Actual: 4 classes — Quad_Cat 0=sad, 1=calm, 2=angry, 3=happy.",
     "VREED uses circumplex valence×arousal quadrants."),
    ("Pupil dilation feature",
     "Spec: pupil_dilation_mm as 1st input feature.",
     "Actual: VREED has no pupil size column. Using Num_of_Blink as proxy",
     "(blink count correlates with arousal, a pupil-size surrogate)."),
    ("Attention model labels",
     "Spec: derive labels from gaze velocity of angular gaze vectors.",
     "Actual: No gaze vectors in dataset. Derive pseudo-labels from",
     "segmentation masks (pupil/iris area ratio + eye visibility ratio)."),
    ("Gaze model architecture",
     "Spec: ResNet-18 regression on gaze angle labels.",
     "Actual: No gaze labels in dataset. Use segmentation model",
     "(MobileNetV2 encoder + segmentation head) → pupil centroid → affine calibration."),
    ("Gaze coordinate space",
     "Spec: normalise OpenEDS pitch/yaw angles to [0,1].",
     "Actual: Pupil centroid (px) in 64×64 crop → affine calibration to screen.",
     "9-point calibration maps crop-space centroid to screen pixels."),
]
for idx, (title, *lines) in enumerate(adaptations, 1):
    print(f"  [{idx}] {title}")
    for line in lines:
        print(f"       {line}")
    print()

print(SEPARATOR)
print("STATUS: All datasets confirmed. Paths verified. Model inputs/outputs defined.")
print("Next step → attention_label_generator.py (Phase 1, File #3)")
print(SEPARATOR)

In [ ]:
# ─── CELL 16 — Confirm saved_models/ directory exists ────────────────────────
for d in ["saved_models", "models", "inference", "tools",
          "data/webcam_finetune"]:
    path = ROOT / d
    path.mkdir(parents=True, exist_ok=True)

log.info("All output directories confirmed / created.")
print("Output directories ready:")
for d in sorted((ROOT / "saved_models").parent.iterdir()):
    if d.is_dir():
        print(f"  {d.relative_to(ROOT)}/")